# 📘 Guía de Estudio 2: Compresión Temporal y Proyector MoE (Mixture of Experts)

En esta segunda guía interactiva, aprenderás:
1. Cómo convertimos coordenadas físicas a la dimensión del LLM (2560 para Gemma).
2. Cómo el **Soft-Gating MoE** pondera la importancia de cada stream (por ejemplo, ignorando la mano si está oculta).
3. Cómo el **Sliding Perceiver Resampler** comprime secuencias largas en un número fijo de tokens visuales.

---

### 1. El Reto: Reducir y Adaptar Dimensiones

El LLM (Gemma 4 E2B) no entiende coordenadas $(x, y, z)$. Él entiende vectores densos de **2560 dimensiones** (los llamados *tokens embeddings*).

Además, una seña puede durar 30 frames (1 segundo) o 150 frames (5 segundos). Pero los LLMs necesitan un número **fijo y pequeño de tokens** para no saturar su contexto.

Nuestra arquitectura hace lo siguiente:
1. **Proyección Lineal:** Proyecta cada stream (mano, pose, rostro) de forma independiente a la dimensión intermedia (ej. 256).
2. **MoE Router (Mixture of Experts):** Calcula dinámicamente qué stream es relevante en cada instante temporal.
3. **Sliding Perceiver Resampler:** Aplica atención cruzada para resumir $T$ frames a solo $M$ tokens visuales fijos (ej. 4 tokens), y los proyecta finalmente a la dimensión del LLM (2560).

### 2. Inspeccionando el LightSignMoEProjector

Vamos a instanciar el proyector que está definido en `src/models/landmark_encoder.py` y ver cómo fluyen los datos a través de él.

In [ ]:
import sys
import os
import torch

# Asegurar que la raíz del proyecto está en el path
sys.path.append(os.path.abspath("../../"))
from src.models.landmark_encoder import LandmarkEncoder

# Instanciamos el proyector con la dimensión de Gemma 4 E2B (2560)
projector = LandmarkEncoder(d_llm=2560)
print(projector)

### 📝 Ejercicio 2: Pasar un Batch de Landmarks simulado y Validar la Salida

Vamos a simular una seña de 60 frames con un batch de tamaño 2. 
Completa el script para:
1. Definir tensores simulados con las formas correctas.
2. Alimentarlos al proyector.
3. Validar que la forma final de salida sea exactamente `[Batch, 4, 2560]` (donde 4 es el número de tokens visuales de salida).

In [ ]:
import torch

B = 2   # Batch size
T = 60  # 60 frames de video

# TODO: Completa las dimensiones correctas usando torch.randn
# Tip: Mano tiene 21 landmarks, rostro 468, pose 33, y todos tienen x, y, z (3 coordenadas)
streams = {
    "left_hand": torch.randn(B, T, 21, 3),
    "right_hand": None, # Reemplaza None por el shape correcto
    "face": None,       # Reemplaza None por el shape correcto
    "pose": None        # Reemplaza None por el shape correcto
}

# Correr el proyector
try:
    output = projector(streams)
    print(f"[+] ¡Éxito! Shape final de los tokens visuales: {list(output.shape)}")
    assert list(output.shape) == [B, 4, 2560], "Dimensiones incorrectas. Deben ser [2, 4, 2560]"
except Exception as e:
    print(f"[-] Error durante la ejecución: {e}")

### Solución sugerida para validar las dimensiones:
```python
streams = {
    "left_hand": torch.randn(B, T, 21, 3),
    "right_hand": torch.randn(B, T, 21, 3),
    "face": torch.randn(B, T, 468, 3),
    "pose": torch.randn(B, T, 33, 3)
}
```